BIDIRECTIONAL LSTM: GUARDARE IL PASSATO PER PREDIRE IL FUTURO.

Per la preveggenza del linguaggio
Permettono alla macchina di leggere il testo come un organismo completo dove il futuro spiega il passato (in senso del linguaggio non di previsione)

La bidirectionl LSTM è una LSTM che analizza la stessa sequenza in due direzioni
LSTM forward
token 1 -> token 2 -> token 3 -> token 4
LSTM backward
token 1 <- token 2 <- token 3 >- token 4
Poi le informazioni ottenute dalle due LSTM vengono combinate.
Il concetto è questo:
una LSTM normale, quando elabora un token, utilizza iò che è venuto prima, una bidirectional LSTM può costruire la rappresentazione usando sia ciò che precede sia ciò che segue quel token

Prendiamo una frase:
"il film che ha visto ieri era davvero fantastico"
con una LSTM normale l'informazione procede da sinestra verso destra
Con una bidirectional LTSM invce abbiamo contemporaneamente da sinistra verso destra e da destra verso sinistra.

Quindi vengono costruiti due stati ed uno nascosto dato dalla somma dei due.
 - hidden state forward: forward layer: legge la frase dall'inizio alla fine ed accumola memoria del passato
 - hidden state backward: backword layer: una copia speculare che riceve la sequenza invertita per catturare il contesto futuro
 - Stato nascosto combianto: ad ogni passo temporale l'output finale è la fusione degli stati dei due layer
che vengono successivamente combinati
Il modello possiede una comprensione completa del contesto circostante per ogni singolo token.

Questo è particolarmete utile in NPL perchè spesso, il significato di una parola, dipende non soltanto da quello che c'è prima, ma anche da ciò che c'è dopo.
Esempio:
"Ho aperto un conto presso la banca" oppure "ci siamo seduti sulla banca del fiume"
Quando elaboriamo la parola "banca", il contesto successivo può aiutare e moltissimo per determinare il significato.
Una rete può utilizzare parole precedenti + parole successive
ha quindi più informazione disponibile.
Il significato di una parola dipende spesso da ciò che viene detto dopo.

Matematicamente quello che succedde è un'operazione di concatenazione, tra i due layer quello che viene da sinistra e quello che viene da destra.
Questo raddoppia la dimensione dello spazio delle feature.

Con Keras è molti semplice implementare una BidirectionalLSTM, si prende una normale LSTM e la si avvolge nel layer bidirectional: layers.Bidirectional(layers.LSTM(64))
C'è però un dettaglio interessante, una LSTM normale con layers(LSTM(64)) restituisce 64 valori, una bidirectionl layers(Bidirectional(layers.LSTM(64))) restituisce 64+64=128 valori
Quindi una bidirectional comporta più parametri e più calcolo, perchè stai utilizzando due reti ricorrenti. Più informazioni ma anche un maggior costo computazionale.

Sintassi e Parametri
- Wrapper: la classe 'layers.bidirectionl che riceve come primo argomento l'istanza del layer ricorrente
- Merge Mode: parametro che definisce come unire i due flussi (concat, sum, mul, ave)
- Dimensionalità: ricordate che con 'concat' (defaul), il numero di unità in uscita raddoppia rispetto al layer base. 
- Compatibilità: Keras gestisce automaticamente l'invervsione della sequenza per il layer backward.

Il Concetto di Merge Mode
Straetgie di fusione informativa
Sebbene la concatenzazione sia lo standard, in casi di memoria limitata o task specifici o di voler mantenere una dimensinalità fissa per i layer successivi, si può optare per la media (ave) o la somma element-wise.
La scelta influenza direttamente la forma del tensore che verrà passato al layer successivo, solitamente un layer Denso o un altro layer ricorrente.

Un'applicazione molto naturale delle BiLSTM è il Named Entity Recognition
Esempio
"Apple ha aperto una nuova sede a Milano"
quando devi classificare ogni token:
Apple -> ORG
Milano -> LOC
può essere utile considerare sia quello che viene prima, sia quello che viene dopo
Esempio:
"Ho mangiato una mela apple" rispetto a "Apple ha presentato un nuovo prodotto"
il contesto circostante aiuta a riconoscere il ruolo della parola 'apple'

Un altro esempio dove la Bi-LSTM è molto utile è nel Part-of-Speech (POS) tagging, decidere se una parola è un sostantivo o un verbo richiede di vedere quello che c'è dopo.

Le BiLSTM pertanto sono molto usate per:
- sentiment analysis: per capire l'enfasi posta a fine periodo
- NER
- POS tagging
- sequence labeling
- classificazione di testi
- traduzione automatica: per creare un vettore rappresentativo dell'intera frase sorgente (code vector)
- riconoscimento vocale: poichè i fonemi sono influenzati dai suoni successivi

Quando NON  usarle?
Supponiamo di voler prevedere la parola successiva:
"il cliente ha ordinato...." 
parola successiva?
Se sei realmente in produzione e la parola successiva non esiste ancora, non puoi utilizzare l'informazione futura. Quindi non puoi fare previsione nel senso stretto del termine.
Una BiLSTM durante l'elaborazione bidirezionale avrebbe bisogno anche della parte successiva della sequenza

Al contrario la BiLSTM per una recensione completa - positivo/negativo va benissimo, perchè al momento della classificazione hai già tutta la recensione.

SimpleRNN → memoria sequenziale semplice
LSTM → memoria controllata con gate → una direzione
GRU → memoria controllata → struttura più compatta
Bidirectional LSTM → due LSTM → una legge avanti → una legge indietro → combina i due contesti

Una Bidirectional LSTM combina due LSTM che elaborano la stessa sequenza in direzioni opposte. In questo modo la rappresentazione di un testo può incorporare informazioni provenienti sia dal contesto precedente sia da quello successivo. È particolarmente utile quando l'intera sequenza è disponibile, mentre non è adatta a una previsione strettamente causale del futuro.


In [1]:
import os

# =================================================================
# 1. SETUP DELL'AMBIENTE (MOTORE DI CALCOLO)
# =================================================================
# Indichiamo a Keras di usare PyTorch come "muscoli" per i calcoli.
# Deve essere la prima operazione assoluta dello script.
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers
import numpy as np

"""

 =================================================================
 BIDIRECTIONAL LSTM (Bi-LSTM) SU DATASET REUTERS
 =================================================================

MAPPA LOGICA DEL CODICE:
1. PREPARAZIONE: Trasformiamo le notizie (testo) in sequenze di numeri.
2. ARCHITETTURA: Costruiamo una rete che legge "avanti e indietro".
3. ADDESTRAMENTO: Il modello impara a collegare parole e argomenti.
4. TEST REALE: Proviamo il modello su news mai viste, decodificando i numeri in parole.

VANTAGGIO BI-LSTM: 
In una news come "Il tasso di interesse della FED...", la parola 'FED' alla fine 
chiarisce che 'tasso' si riferisce alla finanza e non a un animale. La Bi-LSTM
guarda l'intera frase prima di decidere.
"""

# --- CONFIGURAZIONI GLOBALI ---
VOCABOLARIO_SIZE = 10000  # Usiamo solo le 10.000 parole più comuni al mondo
LUNGHEZZA_MAX = 150       # Ogni news viene standardizzata a 150 parole (Taglio o Padding)

# =================================================================
# 2. GESTIONE DEI DATI (IL "CARBURANTE")
# =================================================================
"""
NOTIZIE SUL DATASET REUTERS:
Il dataset è composto da 11.228 lanci d'agenzia (newswires) provenienti 
dalla Reuters, una delle principali agenzie di stampa mondiali.

PUNTI CHIAVE:
- CATEGORIE: 46 argomenti diversi (es. finanze, agricoltura, energia).
- FORMATO: Ogni notizia è già stata pre-elaborata: le parole sono state sostituite 
  da numeri interi che rappresentano la loro frequenza nel dataset.
  - Il numero 10 rappresenta la decima parola più frequente.
  - Questo risparmia tempo nella pulizia dei testi reali.
- SUDDIVISIONE: Circa 8.982 esempi per l'addestramento e 2.246 per il test.
"""

def prepara_dataset():
    """
    Estrae le notizie dal dataset Reuters e le rende uniformi per la rete neurale.
    """
    print("[1] Caricamento Dataset Reuters (46 categorie tematiche)...")
    
    # Caricamento: Keras ci fornisce già gli ID numerici delle parole.
    # Esempio: "apple" diventa 42, "market" diventa 105.
    (x_train, y_train), (x_test, y_test) = keras.datasets.reuters.load_data(num_words=VOCABOLARIO_SIZE)
    
    # INTERAZIONE: pad_sequences trasforma liste di lunghezze diverse in una matrice rettangolare.
    # Se una news ha 50 parole, aggiungiamo 100 zeri (Padding).
    x_train = keras.utils.pad_sequences(x_train, maxlen=LUNGHEZZA_MAX)
    x_test = keras.utils.pad_sequences(x_test, maxlen=LUNGHEZZA_MAX)
    
    # INTERAZIONE: to_categorical trasforma l'indice della categoria (es. 3) in un vettore unitario.
    # Categoria 3 -> [0, 0, 0, 1, 0, ... 0] (46 posizioni totali).
    y_train = keras.utils.to_categorical(y_train, 46)
    y_test = keras.utils.to_categorical(y_test, 46)
    
    return (x_train, y_train), (x_test, y_test)

# =================================================================
# 3. COSTRUZIONE DEL MODELLO (IL "CERVELLO")
# =================================================================
def build_bilstm_classifier():
    """
    Definisce come le informazioni fluiscono dall'input alla decisione finale.
    """
    # Definiamo la forma del dato in ingresso (150 numeri interi)
    inputs = keras.Input(shape=(LUNGHEZZA_MAX,), name="Ingresso_News")

    # EMBEDDING (IL TRADUTTORE)
    # Trasforma ogni numero ID in un vettore di 128 caratteristiche (significato).
    # Interazione: Trasforma INDICI (interi) -> SPAZIO SEMANTICO (float).
    x = layers.Embedding(VOCABOLARIO_SIZE, 128, name="Spazio_Semantico")(inputs)

    # BIDIRECTIONAL LSTM (IL DOPPIO SGUARDO)
    # Layers.Bidirectional avvolge una LSTM standard e ne crea due:
    # 1. Forward LSTM: Legge la frase da sinistra a destra.
    # 2. Backward LSTM: Legge la frase da destra a sinistra.
    # Interazione: Fonde i due contesti in un unico vettore di memoria (64+64 = 128 unità).
    x = layers.Bidirectional(layers.LSTM(64), name="Memoria_Bidirezionale")(x)
    
    # DROPOUT (IL FILTRO ANTI-MEMORIA)
    # Spegne casualmente il 30% dei neuroni per forzare la rete a non imparare i dati a memoria.
    x = layers.Dropout(0.3)(x)
    
    # DENSE (IL RAGIONAMENTO FINALE)
    # Strato con 64 neuroni per elaborare i significati estratti dalla Bi-LSTM.
    x = layers.Dense(64, activation="relu")(x)
    
    # OUTPUT (LA SCELTA FINALE)
    # 46 neuroni (uno per ogni categoria). 'softmax' garantisce che la somma delle probabilità sia 1.
    outputs = layers.Dense(46, activation="softmax", name="Probabilita_Categorie")(x)

    # ASSEMBLAGGIO: Colleghiamo inizio e fine
    model = keras.Model(inputs, outputs, name="Classificatore_BiLSTM_Reuters")
    
    model.compile(
        optimizer="adamw",           # Algoritmo che aggiorna i pesi (il più moderno nel 2026)
        loss="categorical_crossentropy", # Funzione che punisce gli errori di categoria
        metrics=["accuracy"]         # Vogliamo vedere la percentuale di risposte corrette
    )
    return model

# =================================================================
# 4. ESECUZIONE E TEST REAL-TIME
# =================================================================
def main():
    print("-" * 60)
    print("DEMO: BI-LSTM ALL'OPERA SULLE NEWS REUTERS")
    print("-" * 60)
    
    # STEP 1: Preparazione dati (Interazione con la memoria RAM)
    (x_train, y_train), (x_test, y_test) = prepara_dataset()
    
    # STEP 2: Creazione architettura (Interazione con Keras/PyTorch)
    model = build_bilstm_classifier()
    model.summary() # Mostra la struttura e il numero di parametri (pesi) da imparare
    
    # STEP 3: Addestramento (Il "Training" effettivo)
    print("\n[HINT] La rete sta leggendo 9.000 notizie per imparare i temi...")
    model.fit(
        x_train, y_train,
        epochs=10,            # Quante volte la rete rilegge tutto il dataset
        batch_size=128,       # Quante news guarda insieme prima di aggiornare i pesi
        validation_split=0.1, # Usa il 10% per auto-valutarsi durante l'apprendimento
        verbose=1             # Mostra la barra di progresso
    )
    
    # STEP 4: COLLAUDO SUL CAMPO (Inference)
    print("\n" + "="*50)
    print("VERIFICA: IL MODELLO ANALIZZA NOTIZIE REALI")
    print("="*50)

    # Strumenti per decodificare: serve per tornare dai numeri alle parole umane
    word_index = keras.datasets.reuters.get_word_index()
    reverse_word_index = dict([(v, k) for (k, v) in word_index.items()])

    # Analizziamo i primi 3 esempi del set di test (mai visti prima dal modello)
    for i in range(3):
        sample = x_test[i:i+1] # Estraiamo una singola riga di dati
        prediction = model.predict(sample, verbose=0)
        
        pred_idx = np.argmax(prediction) # L'indice della probabilità più alta
        real_idx = np.argmax(y_test[i])   # L'indice reale salvato nel dataset
        
        # Ricostruiamo la frase (saltando i primi 3 indici di sistema di Keras)
        testo = ' '.join([reverse_word_index.get(index - 3, '?') for index in x_test[i]])
        testo_pulito = testo.replace('?', '').strip()[-130:] # Prendiamo la parte finale significativa
        
        print(f"\nNEWS #{i+1}: ...{testo_pulito}")
        print(f"-> PREDIZIONE: Categoria {pred_idx} (Confidenza: {np.max(prediction):.2%})")
        print(f"-> REALTA':    Categoria {real_idx}")
        
    print("\n[FINISH] Esperimento completato. La Bi-LSTM ha 'capito' il contesto globale.")

if __name__ == "__main__":
    main()

# =================================================================
# GUIDA TECNICA (FLUSSO LOGICO) PER LO STUDENTE
# =================================================================
# 1. FLUSSO DATI: Notizia (Testo) -> Liste ID -> Matrice Padding -> Embedding (Vettori) 
#    -> Bi-LSTM (Memoria) -> Dense (Decisione) -> Softmax (Probabilità).
# 2. PERCHÉ LA BI-LSTM? Perché nel giornalismo (Reuters), il soggetto di una frase 
#    lunga viene spesso chiarito solo alla fine. Leggere in entrambi i sensi
#    evita che la rete "si perda" durante il tragitto.
# 3. VERSO LA TRADUZIONE: In un modello di traduzione, la Bi-LSTM qui usata 
#    è l'ENCODER: il suo compito è creare una "mappa mentale" perfetta della 
#    frase sorgente prima di passare la palla al generatore di testo (Decoder).

------------------------------------------------------------
DEMO: BI-LSTM ALL'OPERA SULLE NEWS REUTERS
------------------------------------------------------------
[1] Caricamento Dataset Reuters (46 categorie tematiche)...
2110848/2110848 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "Classificatore_BiLSTM_Reuters"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Ingresso_News (InputLayer)      │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Spazio_Semantico (Embedding)    │ (None, 150, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Memoria_Bidirezionale           │ (None, 128)            │        98,816 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Probabilita_Categorie (Dense)   │ (None, 46)             │         2,990 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,390,062 (5.30 MB)

 Trainable params: 1,390,062 (5.30 MB)

 Non-trainable params: 0 (0.00 B)


[HINT] La rete sta leggendo 9.000 notizie per imparare i temi...
Epoch 1/10


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:1076: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, h_n, c_n = torch._VF.lstm(


64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.3775 - loss: 2.5665 - val_accuracy: 0.4561 - val_loss: 2.0276
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5233 - loss: 1.8113 - val_accuracy: 0.5573 - val_loss: 1.7522
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5791 - loss: 1.6210 - val_accuracy: 0.5762 - val_loss: 1.6983
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6015 - loss: 1.5411 - val_accuracy: 0.5918 - val_loss: 1.6420
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6305 - loss: 1.4176 - val_accuracy: 0.6118 - val_loss: 1.5800
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.6709 - loss: 1.2871 - val_accuracy: 0.6073 - val_loss: 1.5992
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.6849 - loss: 1.2066 - val_accuracy: 0.6107 - val_loss: 1.6402
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.7046 - loss: 1.0993 - val_accuracy: 0.6218 - val_loss: 1.